# 06 Transfer Learning for Object Detection | نقل التعلم لكشف الأشياء

## 📚 Learning Objectives | أهداف التعلم

By completing this notebook (~20 min), you will:
- Understand how **object detection** uses a **pre-trained backbone** + **detection head**
- Use a **pre-trained CNN** as a feature extractor and add a **simple classification head** on top (simplified “detection” setup)
- See why we use transfer learning for detection instead of training from scratch

---

## 🌍 Real life | في الواقع

**Where is this used?** Object detection (localize + classify) is used in **autonomous driving**, **surveillance**, and **retail** (shelf monitoring).

**In this notebook we use** a **pre-trained backbone** (e.g. MobileNetV2) to extract features, then add a **head** for classification. We use **transfer learning for detection** (instead of training a detector from scratch) **because** the backbone already learned good visual features; we only train the head (or fine-tune last layers) with less data.

---

**Before starting:** Run the imports cell below. Full object detection (bounding boxes) uses libraries like TensorFlow Object Detection API; here we show the **backbone + head** idea in ~20 min.

## Theory (short) | النظرية

- **Object detection:** Find **where** objects are (bounding boxes) and **what** they are (class).
- **Typical pipeline:** Pre-trained **backbone** (e.g. ResNet, MobileNet) → **neck** (e.g. FPN) → **detection head** (boxes + classes). YOLO, SSD, Faster R-CNN follow this idea.
- **Transfer learning:** Backbone is pre-trained on ImageNet; we freeze or fine-tune it and train the detection head on our dataset.
- **We use a pre-trained backbone** instead of training from scratch so we need less data and time; the head learns “where” and “what” on top of good features.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** TensorFlow/Keras, NumPy. We use **MNIST resized to 96×96 RGB** (as in 05_transfer_learning_cnns) so the notebook runs without an object-detection dataset.

**Outputs:** Model summary (backbone + head), training loss/accuracy for 2 epochs, and test accuracy. (Full detection would output bounding boxes; here we do **image-level classification** to show the backbone+head pattern.)

## Step 1: Imports and load pre-trained backbone (we use MobileNetV2 as backbone instead of training from scratch)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    backbone = keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
    backbone.trainable = False
    print("Backbone (frozen) params:", backbone.count_params())
else:
    print("Install TensorFlow: pip install tensorflow")

## Step 2: Add classification head (in full detection we would add a head that outputs boxes + classes)

In [ ]:
if HAS_TF:
    inp = keras.Input(shape=(96, 96, 3))
    x = backbone(inp)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, x)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("Model: backbone + global pool + Dense(10). For real detection, head would output boxes + classes.")

## Step 3: Prepare data (MNIST as 96×96 RGB) and train head (2 epochs)

In [ ]:
if HAS_TF:
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = tf.image.resize(x_train[..., np.newaxis], (96, 96))
    x_test = tf.image.resize(x_test[..., np.newaxis], (96, 96))
    x_train = tf.repeat(x_train, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_test = tf.repeat(x_test, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_train, y_train = x_train[:5000], y_train[:5000]
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=64, verbose=1)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Test accuracy: %.4f" % acc)

## ✅ Summary | الملخص

**What you did:** Used a pre-trained backbone (MobileNetV2) + a classification head, trained only the head on MNIST (resized), and saw how transfer learning applies to a detection-style setup.

**In real life you'd also:** Use a real detection dataset (e.g. COCO), add a head that outputs bounding boxes and classes, and use TensorFlow Object Detection API or similar.

**The main idea:** Object detection often uses a pre-trained backbone + a detection head; transfer learning lets us train the head (and optionally fine-tune the backbone) with limited data.

**Next:** `05_transfer_learning_cnns` does transfer learning for classification; for full detection pipelines see TensorFlow Object Detection API.